# 18. 時間帯選別とコスト感度
出典: FX (2).ipynb、セルindex [37, 38]。保存出力は results/imported_20260909/ を参照。
研究履歴です。実行順・Notebook内変数・元の価格CSVに依存し、エラーが出たコードも保存しています。
自動判定の文言は元実験の判定であり、監査済みの結論ではありません。全セル一括実行は再現手順ではありません。
[USER_HOME] は匿名化した元のパスです。元Notebook内の案内や依頼文は研究資料として保持しています。


## 元セルindex 37


In [ ]:
# ============================================================
# USD/JPY 15m
# NESTED SESSION POLICY SELECTION
# + COST / SLIPPAGE STRESS TEST
#
# ============================================================
#
# 目的
# ------------------------------------------------------------
#
# 前回見つかったTime-of-Day差:
#
# UTC 21-24  -> 強い
# UTC 13-21  -> 強い
# UTC 00-08  -> ほぼBreak-even
# UTC 08-13  -> 弱い
#
# が「後付け」ではなく、
# Validationだけで選択して翌年OOSでも有効か検証する。
#
#
# 比較:
#
# BASELINE
#   ValidationでConfidence Thresholdだけ選択
#   Session = ALL固定
#
# SESSION MODEL
#   Validationで
#   Threshold + Session Policyを同時選択
#
#
# Test年では一切変更しない。
#
#
# さらにTest取引に対して
#
# 0.004%
# 0.006%
# 0.008%
# 0.010%
#
# のCost Stress Testを行う。
#
# ============================================================


from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score


# ============================================================
# 1. 設定
# ============================================================

CSV_PATH = (
    Path.cwd()
    / "dukascopy_usdjpy"
    / "usdjpy_15m_2016_2026.csv"
)


RANDOM_STATE = 42

N_ESTIMATORS = 250


# Validationで戦略選択するときの基準Cost
BASE_COST = 0.00004


# Test後のStress Test
COST_LEVELS = [

    0.00004,   # 0.004%
    0.00006,   # 0.006%
    0.00008,   # 0.008%
    0.00010,   # 0.010%

]


THRESHOLDS = [

    0.50,
    0.52,
    0.54,
    0.55,
    0.56,
    0.58,
    0.60,
    0.62,
    0.65,

]


# Session候補
#
# 増やしすぎるとValidation自体が
# 過学習しやすくなるので4種類だけ。
SESSION_POLICIES = [

    "ALL",

    "EXCLUDE_08_13",

    "UTC_13_24",

    "UTC_21_24",

]


MIN_TRAIN_YEARS = 3

MIN_TRAIN_ROWS = 5000

MIN_EVAL_ROWS = 100


# Validation上で最低100 trades
# 少数サンプルでSessionを選ばないため。
MIN_VALIDATION_TRADES = 100


OUTPUT_DIR = (

    Path.cwd()
    /
    (
        "session_nested_cost_"
        +
        datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )
    )

)

OUTPUT_DIR.mkdir(
    exist_ok=False
)


# ============================================================
# 2. データ読み込み
# ============================================================

df = pd.read_csv(

    CSV_PATH,

    index_col=0,

)


df.index = pd.to_datetime(

    df.index,

    utc=True,

)


df.columns = [

    c.lower()

    for c in df.columns

]


required_columns = [

    "open",
    "high",
    "low",
    "close",

]


missing_columns = [

    c

    for c in required_columns

    if c not in df.columns

]


if missing_columns:

    raise ValueError(

        f"Missing OHLC columns: {missing_columns}"

    )


df = (

    df[
        required_columns
    ]

    .apply(
        pd.to_numeric,
        errors="raise"
    )

    .sort_index()

)


if not df.index.is_unique:

    raise ValueError(

        "Timestamp duplicated."

    )


print(
    "===================================="
)

print(
    "DATA"
)

print(
    "===================================="
)

print(
    "Rows:",
    len(df)
)

print(
    "Period:",
    df.index.min(),
    "->",
    df.index.max()
)


# ============================================================
# 3. RSI
# ============================================================

def calculate_rsi(
    close,
    period=14,
):

    delta = (
        close.diff()
    )


    gain = (
        delta.clip(
            lower=0
        )
    )


    loss = (
        -delta.clip(
            upper=0
        )
    )


    avg_gain = (
        gain
        .rolling(period)
        .mean()
    )


    avg_loss = (
        loss
        .rolling(period)
        .mean()
    )


    rs = (

        avg_gain

        /

        avg_loss.replace(
            0,
            np.nan
        )

    )


    return (

        100

        -

        100
        /
        (
            1 + rs
        )

    )


# ============================================================
# 4. Feature Engineering
# ============================================================

FEATURES = [

    "return_1",
    "return_2",
    "return_4",
    "return_8",
    "return_16",

    "vol_4",
    "vol_8",
    "vol_16",
    "vol_32",

    "ma5_distance",
    "ma5_slope",

    "ma10_distance",
    "ma10_slope",

    "ma20_distance",
    "ma20_slope",

    "ma50_distance",
    "ma50_slope",

    "ma100_distance",
    "ma100_slope",

    "body",
    "upper_wick",
    "lower_wick",
    "range_pct",

    "rsi14",
    "atr14",

    "distance_high_16",
    "distance_low_16",

    "hour_sin",
    "hour_cos",

    "weekday",

]


def make_features(
    bars
):

    x = (
        bars.copy()
    )


    # --------------------------------------------------------
    # Return
    # --------------------------------------------------------

    for n in [

        1,
        2,
        4,
        8,
        16,

    ]:

        x[
            f"return_{n}"
        ] = (

            x[
                "close"
            ]

            .pct_change(n)

        )


    # --------------------------------------------------------
    # Volatility
    # --------------------------------------------------------

    for n in [

        4,
        8,
        16,
        32,

    ]:

        x[
            f"vol_{n}"
        ] = (

            x[
                "return_1"
            ]

            .rolling(n)

            .std()

        )


    # --------------------------------------------------------
    # Moving Average
    # --------------------------------------------------------

    for period in [

        5,
        10,
        20,
        50,
        100,

    ]:

        ma = (

            x[
                "close"
            ]

            .rolling(period)

            .mean()

        )


        x[
            f"ma{period}_distance"
        ] = (

            x[
                "close"
            ]
            /
            ma
            -
            1

        )


        x[
            f"ma{period}_slope"
        ] = (

            ma.pct_change()

        )


    # --------------------------------------------------------
    # Candle
    # --------------------------------------------------------

    candle_range = (

        x[
            "high"
        ]

        -

        x[
            "low"
        ]

    ).replace(

        0,

        np.nan

    )


    x[
        "body"
    ] = (

        x[
            "close"
        ]

        -

        x[
            "open"
        ]

    ) / candle_range


    x[
        "upper_wick"
    ] = (

        x[
            "high"
        ]

        -

        x[
            [
                "open",
                "close",
            ]
        ]

        .max(
            axis=1
        )

    ) / candle_range


    x[
        "lower_wick"
    ] = (

        x[
            [
                "open",
                "close",
            ]
        ]

        .min(
            axis=1
        )

        -

        x[
            "low"
        ]

    ) / candle_range


    x[
        "range_pct"
    ] = (

        x[
            "high"
        ]

        -

        x[
            "low"
        ]

    ) / x[
        "close"
    ]


    # --------------------------------------------------------
    # RSI
    # --------------------------------------------------------

    x[
        "rsi14"
    ] = (

        calculate_rsi(

            x[
                "close"
            ],

            14,

        )

        /

        100

    )


    # --------------------------------------------------------
    # ATR
    # --------------------------------------------------------

    previous_close = (

        x[
            "close"
        ]

        .shift(1)

    )


    true_range = (

        pd.concat(

            [

                x[
                    "high"
                ]
                -
                x[
                    "low"
                ],

                (
                    x[
                        "high"
                    ]
                    -
                    previous_close
                ).abs(),

                (
                    x[
                        "low"
                    ]
                    -
                    previous_close
                ).abs(),

            ],

            axis=1,

        )

        .max(
            axis=1
        )

    )


    x[
        "atr14"
    ] = (

        true_range

        .rolling(14)

        .mean()

        /

        x[
            "close"
        ]

    )


    # --------------------------------------------------------
    # Distance to recent High / Low
    # --------------------------------------------------------

    high16 = (

        x[
            "high"
        ]

        .rolling(16)

        .max()

    )


    low16 = (

        x[
            "low"
        ]

        .rolling(16)

        .min()

    )


    x[
        "distance_high_16"
    ] = (

        high16

        -

        x[
            "close"
        ]

    ) / x[
        "close"
    ]


    x[
        "distance_low_16"
    ] = (

        x[
            "close"
        ]

        -

        low16

    ) / x[
        "close"
    ]


    # --------------------------------------------------------
    # Time
    # --------------------------------------------------------

    hour = (

        x.index.hour

        +

        x.index.minute
        /
        60

    )


    x[
        "hour_sin"
    ] = np.sin(

        2
        *
        np.pi
        *
        hour
        /
        24

    )


    x[
        "hour_cos"
    ] = np.cos(

        2
        *
        np.pi
        *
        hour
        /
        24

    )


    x[
        "weekday"
    ] = (

        x.index.dayofweek

        /

        4

    )


    return x


# ============================================================
# 5. Entry / Exit
#
# timestamp = bar OPEN time
#
# t bar確定
# ↓
# t+1 Open Entry
# ↓
# t+2 Close Exit
#
# Entry -> Exit = 30分
# ============================================================

def prepare_data(
    bars
):

    frame = (

        make_features(
            bars
        )

        .replace(

            [
                np.inf,
                -np.inf,
            ],

            np.nan,

        )

    )


    times = pd.Series(

        bars.index,

        index=
            bars.index,

    )


    frame[
        "entry_time"
    ] = (

        times.shift(-1)

    )


    frame[
        "label_end"
    ] = (

        times.shift(-2)

        +

        pd.Timedelta(
            minutes=15
        )

    )


    frame[
        "entry_price"
    ] = (

        bars[
            "open"
        ]

        .shift(-1)

    )


    frame[
        "exit_price"
    ] = (

        bars[
            "close"
        ]

        .shift(-2)

    )


    frame[
        "future_return"
    ] = (

        frame[
            "exit_price"
        ]

        /

        frame[
            "entry_price"
        ]

        -

        1

    )


    frame[
        "target"
    ] = (

        frame[
            "future_return"
        ]

        >
        0

    ).astype(
        int
    )


    # --------------------------------------------------------
    # 欠測 / Weekend等をまたがない
    # 本当に15分→30分の連続足だけ
    # --------------------------------------------------------

    continuous = (

        (
            times.shift(-1)
            -
            times
        )

        ==

        pd.Timedelta(
            minutes=15
        )

    ) & (

        (
            times.shift(-2)
            -
            times
        )

        ==

        pd.Timedelta(
            minutes=30
        )

    )


    frame = (

        frame.loc[
            continuous
        ]

        .dropna(

            subset=
                FEATURES
                +
                [
                    "entry_time",
                    "label_end",
                    "entry_price",
                    "exit_price",
                    "future_return",
                ]

        )

        .copy()

    )


    return frame


data = (
    prepare_data(
        df
    )
)


print()

print(
    "Usable rows:",
    len(data)
)


# ============================================================
# 6. Random Forest
# ============================================================

def build_model():

    return (

        RandomForestClassifier(

            n_estimators=
                N_ESTIMATORS,

            max_depth=
                8,

            min_samples_leaf=
                30,

            max_features=
                "sqrt",

            class_weight=
                "balanced",

            random_state=
                RANDOM_STATE,

            n_jobs=
                -1,

        )

    )


# ============================================================
# 7. Prediction Frame
# ============================================================

def predict_frame(
    model,
    frame
):

    probability = (

        model.predict_proba(

            frame[
                FEATURES
            ]

        )

    )


    classes = list(

        model.classes_

    )


    if (
        0 not in classes
        or
        1 not in classes
    ):

        raise ValueError(

            "Training set must contain both classes."

        )


    p_up = (

        probability[
            :,
            classes.index(
                1
            )
        ]

    )


    result = (

        frame[
            [
                "entry_time",
                "label_end",
                "entry_price",
                "exit_price",
                "future_return",
            ]
        ]

        .copy()

    )


    result[
        "p_up"
    ] = (
        p_up
    )


    result[
        "confidence"
    ] = np.maximum(

        p_up,

        1
        -
        p_up

    )


    result[
        "direction"
    ] = np.where(

        p_up
        >=
        0.5,

        "BUY",

        "SELL",

    )


    result[
        "direction_correct"
    ] = (

        (
            p_up
            >=
            0.5
        )

        ==

        (
            frame[
                "future_return"
            ]
            >
            0
        )

    )


    result[
        "gross_return"
    ] = (

        frame[
            "future_return"
        ]

        *

        np.where(

            p_up
            >=
            0.5,

            1,

            -1,

        )

    )


    return result


# ============================================================
# 8. Session Policy
# ============================================================

def apply_session_policy(
    frame,
    policy
):

    if frame.empty:

        return (
            frame.copy()
        )


    hour = (

        frame[
            "entry_time"
        ]

        .dt.hour

    )


    if (
        policy
        ==
        "ALL"
    ):

        mask = (
            np.ones(
                len(frame),
                dtype=bool
            )
        )


    elif (
        policy
        ==
        "EXCLUDE_08_13"
    ):

        mask = ~(

            (
                hour
                >=
                8
            )

            &

            (
                hour
                <
                13
            )

        )


    elif (
        policy
        ==
        "UTC_13_24"
    ):

        mask = (

            hour
            >=
            13

        )


    elif (
        policy
        ==
        "UTC_21_24"
    ):

        mask = (

            hour
            >=
            21

        )


    else:

        raise ValueError(

            f"Unknown session policy: {policy}"

        )


    return (

        frame.loc[
            mask
        ]

        .copy()

    )


# ============================================================
# 9. Trade Selection
#
# IMPORTANT FIX:
#
# previous Exit = 00:45 Close
# next Entry     = 00:45 Open
#
# これは同じbarのOpen/Closeなので重複。
#
# entry_time == previous label_end
# も禁止する。
# ============================================================

def select_trades(
    predictions,
    threshold,
    session_policy,
    cost,
):

    candidates = (

        predictions.loc[

            predictions[
                "confidence"
            ]

            >=

            threshold

        ]

        .sort_index()

        .copy()

    )


    candidates = (

        apply_session_policy(

            candidates,

            session_policy,

        )

    )


    selected_indices = []


    last_exit_time = None


    for row in (

        candidates.itertuples()

    ):


        # ------------------------------------
        # 完全に前ポジション終了後だけEntry
        #
        # equalityも禁止
        # ------------------------------------

        if (

            last_exit_time
            is not None

            and

            row.entry_time
            <=
            last_exit_time

        ):

            continue


        selected_indices.append(

            row.Index

        )


        last_exit_time = (

            row.label_end

        )


    trades = (

        candidates.loc[
            selected_indices
        ]

        .copy()

    )


    trades[
        "net_return"
    ] = (

        trades[
            "gross_return"
        ]

        -

        cost

    )


    trades[
        "threshold"
    ] = (
        threshold
    )


    trades[
        "session_policy"
    ] = (
        session_policy
    )


    trades.index.name = (
        "signal_time"
    )


    return trades


# ============================================================
# 10. Performance
# ============================================================

def profit_factor(
    returns
):

    r = np.asarray(

        returns,

        dtype=float,

    )


    gains = (

        r[
            r > 0
        ]

        .sum()

    )


    losses = (

        -r[
            r < 0
        ]

        .sum()

    )


    if (
        losses
        >
        0
    ):

        return (

            gains
            /
            losses

        )


    if (
        gains
        >
        0
    ):

        return (
            np.inf
        )


    return (
        np.nan
    )


def strategy_stats(
    returns
):

    r = np.asarray(

        returns,

        dtype=float,

    )


    if len(
        r
    ) == 0:

        return {

            "trades":
                0,

            "win_rate":
                np.nan,

            "avg_return":
                np.nan,

            "median_return":
                np.nan,

            "profit_factor":
                np.nan,

            "max_dd":
                np.nan,

            "growth":
                np.nan,

        }


    equity = np.r_[

        1.0,

        np.cumprod(

            1
            +
            r

        )

    ]


    peak = (

        np.maximum.accumulate(

            equity

        )

    )


    dd = (

        equity

        /
        peak

        -

        1

    )


    return {

        "trades":
            len(r),

        "win_rate":
            (
                r > 0
            ).mean(),

        "avg_return":
            r.mean(),

        "median_return":
            np.median(
                r
            ),

        "profit_factor":
            profit_factor(
                r
            ),

        "max_dd":
            dd.min(),

        "growth":
            equity[-1]
            -
            1,

    }


# ============================================================
# 11. Validation Strategy Search
#
# score =
#
# AvgReturn × sqrt(Trades)
#
# PF最大化だけだと少数tradeが選ばれやすいため
# PFは目的関数にしない。
# ============================================================

def search_strategy(
    validation_predictions,
    session_policies,
):

    rows = []


    best = None


    best_score = (
        -np.inf
    )


    for threshold in (
        THRESHOLDS
    ):


        for policy in (
            session_policies
        ):


            trades = (

                select_trades(

                    validation_predictions,

                    threshold=
                        threshold,

                    session_policy=
                        policy,

                    cost=
                        BASE_COST,

                )

            )


            stats = (

                strategy_stats(

                    trades[
                        "net_return"
                    ]

                )

            )


            eligible = (

                stats[
                    "trades"
                ]

                >=

                MIN_VALIDATION_TRADES

            )


            if (
                eligible
            ):

                score = (

                    stats[
                        "avg_return"
                    ]

                    *

                    np.sqrt(

                        stats[
                            "trades"
                        ]

                    )

                )


            else:

                score = (
                    np.nan
                )


            rows.append(

                {

                    "threshold":
                        threshold,

                    "session_policy":
                        policy,

                    "eligible":
                        eligible,

                    "score":
                        score,

                    **stats,

                }

            )


            if (
                eligible

                and

                score
                >
                best_score
            ):

                best_score = (
                    score
                )


                best = {

                    "threshold":
                        threshold,

                    "session_policy":
                        policy,

                    "score":
                        score,

                    "validation_trades":
                        stats[
                            "trades"
                        ],

                    "validation_avg_return":
                        stats[
                            "avg_return"
                        ],

                    "validation_pf":
                        stats[
                            "profit_factor"
                        ],

                }


    return (

        best,

        pd.DataFrame(
            rows
        ),

    )


# ============================================================
# 12. Nested Walk-Forward
# ============================================================

years = sorted(

    data.index.year.unique()

)


annual_rows = []


baseline_trade_frames = []

session_trade_frames = []


validation_search_frames = []


for test_year in (
    years
):


    validation_year = (

        test_year
        -
        1

    )


    previous_years = [

        y

        for y in years

        if y
        <
        validation_year

    ]


    if (
        len(
            previous_years
        )
        <
        MIN_TRAIN_YEARS
    ):

        continue


    if (
        validation_year
        not in years
    ):

        continue


    validation_start = pd.Timestamp(

        year=
            validation_year,

        month=1,

        day=1,

        tz="UTC",

    )


    test_start = pd.Timestamp(

        year=
            test_year,

        month=1,

        day=1,

        tz="UTC",

    )


    test_end = pd.Timestamp(

        year=
            test_year
            +
            1,

        month=1,

        day=1,

        tz="UTC",

    )


    # ========================================================
    # Train
    #
    # label_end <= Validation開始
    # ========================================================

    train = (

        data.loc[

            (
                data.index
                <
                validation_start
            )

            &

            (
                data[
                    "label_end"
                ]
                <=
                validation_start
            )

        ]

        .copy()

    )


    # ========================================================
    # Validation
    # ========================================================

    validation = (

        data.loc[

            (
                data.index
                >=
                validation_start
            )

            &

            (
                data.index
                <
                test_start
            )

            &

            (
                data[
                    "label_end"
                ]
                <=
                test_start
            )

        ]

        .copy()

    )


    # ========================================================
    # Final Train
    # ========================================================

    final_train = (

        data.loc[

            (
                data.index
                <
                test_start
            )

            &

            (
                data[
                    "label_end"
                ]
                <=
                test_start
            )

        ]

        .copy()

    )


    # ========================================================
    # Test
    # ========================================================

    test = (

        data.loc[

            (
                data.index
                >=
                test_start
            )

            &

            (
                data.index
                <
                test_end
            )

            &

            (
                data[
                    "label_end"
                ]
                <=
                test_end
            )

        ]

        .copy()

    )


    if (

        len(
            train
        )
        <
        MIN_TRAIN_ROWS

        or

        len(
            validation
        )
        <
        MIN_EVAL_ROWS

        or

        len(
            final_train
        )
        <
        MIN_TRAIN_ROWS

        or

        len(
            test
        )
        <
        MIN_EVAL_ROWS

    ):

        continue


    print()

    print(
        "===================================="
    )

    print(
        f"TEST YEAR {test_year}"
    )

    print(
        "===================================="
    )


    # ========================================================
    # Train -> Validation
    # ========================================================

    validation_model = (

        build_model()

    )


    validation_model.fit(

        train[
            FEATURES
        ],

        train[
            "target"
        ],

    )


    validation_predictions = (

        predict_frame(

            validation_model,

            validation,

        )

    )


    # ========================================================
    # BASELINE
    #
    # ThresholdだけValidationで選択
    # Session = ALL固定
    # ========================================================

    (
        baseline_choice,
        baseline_search,
    ) = search_strategy(

        validation_predictions,

        session_policies=[
            "ALL"
        ],

    )


    # ========================================================
    # SESSION STRATEGY
    #
    # Threshold + Session Policy
    # ========================================================

    (
        session_choice,
        session_search,
    ) = search_strategy(

        validation_predictions,

        session_policies=
            SESSION_POLICIES,

    )


    if (
        baseline_choice
        is None

        or

        session_choice
        is None
    ):

        print(
            "No eligible strategy."
        )

        continue


    baseline_search[
        "strategy"
    ] = (
        "BASELINE"
    )


    session_search[
        "strategy"
    ] = (
        "SESSION"
    )


    baseline_search[
        "test_year"
    ] = (
        test_year
    )


    session_search[
        "test_year"
    ] = (
        test_year
    )


    validation_search_frames.extend(

        [
            baseline_search,
            session_search,
        ]

    )


    print(
        "BASE choice:",
        baseline_choice[
            "threshold"
        ],
        baseline_choice[
            "session_policy"
        ]
    )


    print(
        "SESSION choice:",
        session_choice[
            "threshold"
        ],
        session_choice[
            "session_policy"
        ]
    )


    # ========================================================
    # Train + Validationで再学習
    # ========================================================

    final_model = (

        build_model()

    )


    final_model.fit(

        final_train[
            FEATURES
        ],

        final_train[
            "target"
        ],

    )


    test_predictions = (

        predict_frame(

            final_model,

            test,

        )

    )


    test_auc = (

        roc_auc_score(

            test[
                "target"
            ],

            test_predictions[
                "p_up"
            ],

        )

    )


    # ========================================================
    # BASELINE Test
    # ========================================================

    baseline_trades = (

        select_trades(

            test_predictions,

            threshold=
                baseline_choice[
                    "threshold"
                ],

            session_policy=
                "ALL",

            cost=
                BASE_COST,

        )

    )


    baseline_trades[
        "test_year"
    ] = (
        test_year
    )


    baseline_trades[
        "strategy"
    ] = (
        "BASELINE"
    )


    baseline_trade_frames.append(

        baseline_trades

    )


    baseline_stats = (

        strategy_stats(

            baseline_trades[
                "net_return"
            ]

        )

    )


    # ========================================================
    # SESSION Test
    # ========================================================

    session_trades = (

        select_trades(

            test_predictions,

            threshold=
                session_choice[
                    "threshold"
                ],

            session_policy=
                session_choice[
                    "session_policy"
                ],

            cost=
                BASE_COST,

        )

    )


    session_trades[
        "test_year"
    ] = (
        test_year
    )


    session_trades[
        "strategy"
    ] = (
        "SESSION"
    )


    session_trade_frames.append(

        session_trades

    )


    session_stats = (

        strategy_stats(

            session_trades[
                "net_return"
            ]

        )

    )


    annual_rows.append(

        {

            "test_year":
                test_year,

            "test_auc":
                test_auc,


            # BASELINE
            "base_threshold":
                baseline_choice[
                    "threshold"
                ],

            "base_trades":
                baseline_stats[
                    "trades"
                ],

            "base_avg_return":
                baseline_stats[
                    "avg_return"
                ],

            "base_pf":
                baseline_stats[
                    "profit_factor"
                ],

            "base_max_dd":
                baseline_stats[
                    "max_dd"
                ],


            # SESSION
            "session_threshold":
                session_choice[
                    "threshold"
                ],

            "session_policy":
                session_choice[
                    "session_policy"
                ],

            "session_trades":
                session_stats[
                    "trades"
                ],

            "session_avg_return":
                session_stats[
                    "avg_return"
                ],

            "session_pf":
                session_stats[
                    "profit_factor"
                ],

            "session_max_dd":
                session_stats[
                    "max_dd"
                ],

        }

    )


    print(
        "Test AUC:",
        round(
            test_auc,
            4
        )
    )


    print(
        "BASE:",
        baseline_stats[
            "trades"
        ],
        "trades | PF",
        round(
            baseline_stats[
                "profit_factor"
            ],
            3
        ),
        "| Avg",
        round(
            baseline_stats[
                "avg_return"
            ]
            *
            100,
            5
        ),
        "%"
    )


    print(
        "SESSION:",
        session_stats[
            "trades"
        ],
        "trades | PF",
        round(
            session_stats[
                "profit_factor"
            ],
            3
        ),
        "| Avg",
        round(
            session_stats[
                "avg_return"
            ]
            *
            100,
            5
        ),
        "%"
    )


# ============================================================
# 13. 年別比較
# ============================================================

annual_results = (

    pd.DataFrame(
        annual_rows
    )

)


print()

print(
    "===================================="
)

print(
    "ANNUAL BASELINE vs SESSION"
)

print(
    "===================================="
)


annual_show = (

    annual_results.copy()

)


for col in [

    "base_threshold",
    "session_threshold",

    "base_avg_return",
    "session_avg_return",

    "base_max_dd",
    "session_max_dd",

]:

    annual_show[
        col
    ] *= (
        100
    )


print(

    annual_show.to_string(
        index=False
    )

)


# ============================================================
# 14. 全OOS統合
# ============================================================

all_baseline_trades = (

    pd.concat(
        baseline_trade_frames
    )

    .sort_index()

)


all_session_trades = (

    pd.concat(
        session_trade_frames
    )

    .sort_index()

)


baseline_overall = (

    strategy_stats(

        all_baseline_trades[
            "net_return"
        ]

    )

)


session_overall = (

    strategy_stats(

        all_session_trades[
            "net_return"
        ]

    )

)


print()

print(
    "===================================="
)

print(
    "OVERALL OOS"
)

print(
    "===================================="
)


print(
    "BASELINE"
)

print(
    baseline_overall
)


print()

print(
    "SESSION"
)

print(
    session_overall
)


# ============================================================
# 15. Session Policy選択頻度
# ============================================================

print()

print(
    "===================================="
)

print(
    "SESSION POLICY SELECTION"
)

print(
    "===================================="
)


print(

    annual_results[
        "session_policy"
    ]

    .value_counts()

)


print()

print(
    "Threshold"
)


print(

    annual_results[
        "session_threshold"
    ]

    .value_counts()

    .sort_index()

)


# ============================================================
# 16. 年単位でSESSIONがBASEを上回ったか
# ============================================================

annual_results[
    "session_better_avg"
] = (

    annual_results[
        "session_avg_return"
    ]

    >

    annual_results[
        "base_avg_return"
    ]

)


annual_results[
    "session_better_pf"
] = (

    annual_results[
        "session_pf"
    ]

    >

    annual_results[
        "base_pf"
    ]

)


print()

print(
    "===================================="
)

print(
    "SESSION vs BASE STABILITY"
)

print(
    "===================================="
)


print(

    "Avg Return better:",

    annual_results[
        "session_better_avg"
    ].sum(),

    "/",

    len(
        annual_results
    )

)


print(

    "PF better:",

    annual_results[
        "session_better_pf"
    ].sum(),

    "/",

    len(
        annual_results
    )

)


print(

    "Session positive years:",

    (
        annual_results[
            "session_avg_return"
        ]
        >
        0
    ).sum(),

    "/",

    len(
        annual_results
    )

)


print(

    "Session PF > 1 years:",

    (
        annual_results[
            "session_pf"
        ]
        >
        1
    ).sum(),

    "/",

    len(
        annual_results
    )

)


# ============================================================
# 17. Cost Stress Test
#
# 重要:
#
# Session / Thresholdは0.004% Validationで
# すでに固定済み。
#
# Costを変えて再最適化しない。
# ============================================================

cost_rows = []


def run_cost_stress(
    trades,
    strategy_name,
):

    for cost in (
        COST_LEVELS
    ):


        returns = (

            trades[
                "gross_return"
            ]

            -

            cost

        )


        stats = (

            strategy_stats(
                returns
            )

        )


        cost_rows.append(

            {

                "strategy":
                    strategy_name,

                "cost_pct":
                    cost
                    *
                    100,

                **stats,

            }

        )


run_cost_stress(

    all_baseline_trades,

    "BASELINE",

)


run_cost_stress(

    all_session_trades,

    "SESSION",

)


cost_results = (

    pd.DataFrame(
        cost_rows
    )

)


cost_show = (

    cost_results.copy()

)


for col in [

    "win_rate",
    "avg_return",
    "median_return",
    "max_dd",
    "growth",

]:

    cost_show[
        col
    ] *= (
        100
    )


print()

print(
    "===================================="
)

print(
    "COST STRESS TEST"
)

print(
    "===================================="
)


print(

    cost_show.to_string(
        index=False
    )

)


# ============================================================
# 18. 年別Cost Stress
# ============================================================

year_cost_rows = []


for strategy_name, trade_frame in [

    (
        "BASELINE",
        all_baseline_trades,
    ),

    (
        "SESSION",
        all_session_trades,
    ),

]:


    for year, year_trades in (

        trade_frame.groupby(
            "test_year"
        )

    ):


        for cost in (
            COST_LEVELS
        ):


            returns = (

                year_trades[
                    "gross_return"
                ]

                -

                cost

            )


            stats = (

                strategy_stats(
                    returns
                )

            )


            year_cost_rows.append(

                {

                    "strategy":
                        strategy_name,

                    "test_year":
                        year,

                    "cost_pct":
                        cost
                        *
                        100,

                    **stats,

                }

            )


year_cost_results = (

    pd.DataFrame(
        year_cost_rows
    )

)


# ============================================================
# 19. Cost耐性の年数
# ============================================================

cost_stability_rows = []


for (

    strategy_name,
    cost_pct

), group in (

    year_cost_results.groupby(

        [
            "strategy",
            "cost_pct",
        ]

    )

):


    cost_stability_rows.append(

        {

            "strategy":
                strategy_name,

            "cost_pct":
                cost_pct,

            "years":
                len(
                    group
                ),

            "positive_years":
                (
                    group[
                        "avg_return"
                    ]
                    >
                    0
                ).sum(),

            "pf_above_1_years":
                (
                    group[
                        "profit_factor"
                    ]
                    >
                    1
                ).sum(),

            "median_year_pf":
                group[
                    "profit_factor"
                ].median(),

            "minimum_year_pf":
                group[
                    "profit_factor"
                ].min(),

        }

    )


cost_stability = (

    pd.DataFrame(
        cost_stability_rows
    )

)


print()

print(
    "===================================="
)

print(
    "COST YEAR STABILITY"
)

print(
    "===================================="
)


print(

    cost_stability.to_string(
        index=False
    )

)


# ============================================================
# 20. Break-even Cost概算
#
# gross average return ≒
# strategyが耐えられる平均コスト上限
# ============================================================

baseline_gross_mean = (

    all_baseline_trades[
        "gross_return"
    ].mean()

)


session_gross_mean = (

    all_session_trades[
        "gross_return"
    ].mean()

)


print()

print(
    "===================================="
)

print(
    "APPROX BREAK-EVEN COST"
)

print(
    "===================================="
)


print(

    "BASELINE gross mean:",

    baseline_gross_mean
    *
    100,

    "%"

)


print(

    "SESSION gross mean:",

    session_gross_mean
    *
    100,

    "%"

)


print()

print(

    "粗い解釈:"

)


print(

    "この数値付近まで総コストが上がると、"

)


print(

    "平均expectancyは概ね0へ近づきます。"

)


# ============================================================
# 21. Automatic Diagnostic
# ============================================================

print()

print(
    "===================================="
)

print(
    "AUTOMATIC DIAGNOSTIC"
)

print(
    "===================================="
)


session_pf = (

    session_overall[
        "profit_factor"
    ]

)


base_pf = (

    baseline_overall[
        "profit_factor"
    ]

)


better_years = (

    annual_results[
        "session_better_pf"
    ].sum()

)


if (

    session_pf
    >
    base_pf

    and

    better_years
    >=
    4

):

    print(

        "Session selection appears to add OOS value."

    )


else:

    print(

        "Session selection does NOT clearly improve the baseline."

    )

    print(

        "Do not add the Session Filter to the strategy yet."

    )


high_cost_session = (

    cost_results.loc[

        (
            cost_results[
                "strategy"
            ]
            ==
            "SESSION"
        )

        &

        np.isclose(

            cost_results[
                "cost_pct"
            ],

            0.008,

        )

    ]

)


if (
    not high_cost_session.empty
):


    pf_008 = (

        high_cost_session.iloc[0][
            "profit_factor"
        ]

    )


    print()

    print(

        "SESSION PF at 0.008% cost:",

        pf_008

    )


    if (
        pf_008
        >
        1
    ):

        print(

            "Edge survives the 0.008% cost stress."

        )


    else:

        print(

            "Edge disappears around the 0.008% cost stress."

        )


# ============================================================
# 22. Graph - Annual PF
# ============================================================

plt.figure(

    figsize=(
        9,
        5
    )

)


plt.plot(

    annual_results[
        "test_year"
    ],

    annual_results[
        "base_pf"
    ],

    marker="o",

    label="Baseline",

)


plt.plot(

    annual_results[
        "test_year"
    ],

    annual_results[
        "session_pf"
    ],

    marker="o",

    label="Session",

)


plt.axhline(

    1,

    linewidth=1,

)


plt.xlabel(
    "Test Year"
)

plt.ylabel(
    "Profit Factor"
)

plt.title(
    "Nested OOS: Baseline vs Session"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# 23. Graph - Cost vs PF
# ============================================================

plt.figure(

    figsize=(
        9,
        5
    )

)


for strategy_name in [

    "BASELINE",
    "SESSION",

]:

    temp = (

        cost_results.loc[

            cost_results[
                "strategy"
            ]
            ==
            strategy_name

        ]

    )


    plt.plot(

        temp[
            "cost_pct"
        ],

        temp[
            "profit_factor"
        ],

        marker="o",

        label=
            strategy_name,

    )


plt.axhline(

    1,

    linewidth=1,

)


plt.xlabel(
    "Cost (%)"
)

plt.ylabel(
    "Profit Factor"
)

plt.title(
    "OOS Cost Stress"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# 24. 保存
# ============================================================

annual_results.to_csv(

    OUTPUT_DIR
    /
    "annual_baseline_vs_session.csv",

    index=False,

)


all_baseline_trades.to_csv(

    OUTPUT_DIR
    /
    "baseline_oos_trades.csv",

)


all_session_trades.to_csv(

    OUTPUT_DIR
    /
    "session_oos_trades.csv",

)


cost_results.to_csv(

    OUTPUT_DIR
    /
    "cost_stress_overall.csv",

    index=False,

)


year_cost_results.to_csv(

    OUTPUT_DIR
    /
    "cost_stress_yearly.csv",

    index=False,

)


cost_stability.to_csv(

    OUTPUT_DIR
    /
    "cost_stability.csv",

    index=False,

)


if validation_search_frames:

    validation_search = (

        pd.concat(

            validation_search_frames,

            ignore_index=True,

        )

    )


    validation_search.to_csv(

        OUTPUT_DIR
        /
        "validation_strategy_search.csv",

        index=False,

    )


print()

print(
    "===================================="
)

print(
    "FINISHED"
)

print(
    "===================================="
)


print(

    OUTPUT_DIR.resolve()

)


print()

print(
    "特に確認するところ:"
)

print(
    "1. ANNUAL BASELINE vs SESSION"
)

print(
    "2. OVERALL OOS"
)

print(
    "3. SESSION POLICY SELECTION"
)

print(
    "4. SESSION vs BASE STABILITY"
)

print(
    "5. COST STRESS TEST"
)

print(
    "6. COST YEAR STABILITY"
)

print(
    "7. APPROX BREAK-EVEN COST"
)


## 元セルindex 38


In [ ]:
# ============================================================
# USD/JPY 15m
# PURE SESSION VALUE TEST
#
# Corrected Nested Walk-Forward
# Threshold first -> fixed Threshold -> Session selection
#
# ============================================================
#
# 今回の目的
# ------------------------------------------------------------
#
# 前回は
#
#   Threshold + Session
#
# を同時にValidationで選択していた。
#
# そのため、
# 「Sessionが良かったのか」
# 「Threshold変更が良かったのか」
# を完全には分離できなかった。
#
#
# 今回は:
#
# 1. Train
# 2. ValidationでThresholdをALL sessionだけで決定
# 3. Thresholdを固定
# 4. 同じValidationでSession Policyだけを選択
# 5. 翌年Testで完全固定
#
#
# 比較:
#
# BASELINE
#   Threshold固定
#   Session = ALL
#
# SESSION
#   同じThreshold
#   Validationで選んだSession
#
#
# これでSession Filterの純粋な追加価値を調べる。
#
# ------------------------------------------------------------
#
# Entry / Exit
#
# t bar closeでSignal
# t+1 OpenでEntry
# t+2 CloseでExit
#
# Entry -> Exit 約30分
#
# ------------------------------------------------------------
#
# Cost:
# Validation selection = 0.004%
#
# Test後Stress:
# 0.004 / 0.006 / 0.008 / 0.010%
#
# ============================================================


from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score


# ============================================================
# 1. CONFIG
# ============================================================

CSV_PATH = (
    Path.cwd()
    / "dukascopy_usdjpy"
    / "usdjpy_15m_2016_2026.csv"
)


RANDOM_STATE = 42

N_ESTIMATORS = 250


BASE_COST = 0.00004


COST_LEVELS = [

    0.00004,   # 0.004%
    0.00006,   # 0.006%
    0.00008,   # 0.008%
    0.00010,   # 0.010%

]


THRESHOLDS = [

    0.50,
    0.52,
    0.54,
    0.55,
    0.56,
    0.58,
    0.60,
    0.62,
    0.65,

]


SESSION_POLICIES = [

    "ALL",

    "EXCLUDE_08_13",

    "UTC_13_24",

    "UTC_21_24",

]


MIN_TRAIN_YEARS = 3

MIN_TRAIN_ROWS = 5000

MIN_EVAL_ROWS = 100

MIN_VALIDATION_TRADES = 100


OUTPUT_DIR = (

    Path.cwd()
    /
    (
        "pure_session_nested_"
        +
        datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )
    )

)


OUTPUT_DIR.mkdir(
    exist_ok=False
)


# ============================================================
# 2. LOAD DATA
# ============================================================

df = pd.read_csv(

    CSV_PATH,

    index_col=0,

)


df.index = pd.to_datetime(

    df.index,

    utc=True,

)


df.columns = [

    c.lower()

    for c in df.columns

]


required_columns = [

    "open",
    "high",
    "low",
    "close",

]


missing = [

    c

    for c in required_columns

    if c not in df.columns

]


if missing:

    raise ValueError(

        f"Missing OHLC columns: {missing}"

    )


df = (

    df[
        required_columns
    ]

    .apply(
        pd.to_numeric,
        errors="raise"
    )

    .sort_index()

)


if not df.index.is_unique:

    raise ValueError(

        "Duplicate timestamps found."

    )


print(
    "===================================="
)

print(
    "DATA"
)

print(
    "===================================="
)

print(
    "Rows:",
    len(df)
)

print(
    "Period:",
    df.index.min(),
    "->",
    df.index.max()
)


# ============================================================
# 3. RSI
# ============================================================

def calculate_rsi(
    close,
    period=14,
):

    delta = (
        close.diff()
    )


    gain = (
        delta.clip(
            lower=0
        )
    )


    loss = (
        -delta.clip(
            upper=0
        )
    )


    avg_gain = (

        gain

        .rolling(period)

        .mean()

    )


    avg_loss = (

        loss

        .rolling(period)

        .mean()

    )


    rs = (

        avg_gain

        /

        avg_loss.replace(
            0,
            np.nan
        )

    )


    return (

        100

        -

        100
        /
        (
            1 + rs
        )

    )


# ============================================================
# 4. FEATURES
# ============================================================

FEATURES = [

    "return_1",
    "return_2",
    "return_4",
    "return_8",
    "return_16",

    "vol_4",
    "vol_8",
    "vol_16",
    "vol_32",

    "ma5_distance",
    "ma5_slope",

    "ma10_distance",
    "ma10_slope",

    "ma20_distance",
    "ma20_slope",

    "ma50_distance",
    "ma50_slope",

    "ma100_distance",
    "ma100_slope",

    "body",
    "upper_wick",
    "lower_wick",
    "range_pct",

    "rsi14",
    "atr14",

    "distance_high_16",
    "distance_low_16",

    "hour_sin",
    "hour_cos",

    "weekday",

]


def make_features(
    bars
):

    x = (
        bars.copy()
    )


    # --------------------------------------------------------
    # RETURN
    # --------------------------------------------------------

    for n in [

        1,
        2,
        4,
        8,
        16,

    ]:

        x[
            f"return_{n}"
        ] = (

            x[
                "close"
            ]

            .pct_change(n)

        )


    # --------------------------------------------------------
    # VOLATILITY
    # --------------------------------------------------------

    for n in [

        4,
        8,
        16,
        32,

    ]:

        x[
            f"vol_{n}"
        ] = (

            x[
                "return_1"
            ]

            .rolling(n)

            .std()

        )


    # --------------------------------------------------------
    # MOVING AVERAGE
    # --------------------------------------------------------

    for period in [

        5,
        10,
        20,
        50,
        100,

    ]:

        ma = (

            x[
                "close"
            ]

            .rolling(period)

            .mean()

        )


        x[
            f"ma{period}_distance"
        ] = (

            x[
                "close"
            ]

            /

            ma

            -

            1

        )


        x[
            f"ma{period}_slope"
        ] = (

            ma.pct_change()

        )


    # --------------------------------------------------------
    # CANDLE
    # --------------------------------------------------------

    candle_range = (

        x[
            "high"
        ]

        -

        x[
            "low"
        ]

    ).replace(

        0,

        np.nan

    )


    x[
        "body"
    ] = (

        x[
            "close"
        ]

        -

        x[
            "open"
        ]

    ) / candle_range


    x[
        "upper_wick"
    ] = (

        x[
            "high"
        ]

        -

        x[
            [
                "open",
                "close",
            ]
        ]

        .max(
            axis=1
        )

    ) / candle_range


    x[
        "lower_wick"
    ] = (

        x[
            [
                "open",
                "close",
            ]
        ]

        .min(
            axis=1
        )

        -

        x[
            "low"
        ]

    ) / candle_range


    x[
        "range_pct"
    ] = (

        x[
            "high"
        ]

        -

        x[
            "low"
        ]

    ) / x[
        "close"
    ]


    # --------------------------------------------------------
    # RSI
    # --------------------------------------------------------

    x[
        "rsi14"
    ] = (

        calculate_rsi(

            x[
                "close"
            ],

            14,

        )

        /

        100

    )


    # --------------------------------------------------------
    # ATR
    # --------------------------------------------------------

    previous_close = (

        x[
            "close"
        ]

        .shift(1)

    )


    true_range = (

        pd.concat(

            [

                x[
                    "high"
                ]
                -
                x[
                    "low"
                ],

                (
                    x[
                        "high"
                    ]
                    -
                    previous_close
                ).abs(),

                (
                    x[
                        "low"
                    ]
                    -
                    previous_close
                ).abs(),

            ],

            axis=1,

        )

        .max(
            axis=1
        )

    )


    x[
        "atr14"
    ] = (

        true_range

        .rolling(14)

        .mean()

        /

        x[
            "close"
        ]

    )


    # --------------------------------------------------------
    # RECENT HIGH / LOW
    # --------------------------------------------------------

    high16 = (

        x[
            "high"
        ]

        .rolling(16)

        .max()

    )


    low16 = (

        x[
            "low"
        ]

        .rolling(16)

        .min()

    )


    x[
        "distance_high_16"
    ] = (

        high16

        -

        x[
            "close"
        ]

    ) / x[
        "close"
    ]


    x[
        "distance_low_16"
    ] = (

        x[
            "close"
        ]

        -

        low16

    ) / x[
        "close"
    ]


    # --------------------------------------------------------
    # TIME
    # --------------------------------------------------------

    hour = (

        x.index.hour

        +

        x.index.minute
        /
        60

    )


    x[
        "hour_sin"
    ] = np.sin(

        2
        *
        np.pi
        *
        hour
        /
        24

    )


    x[
        "hour_cos"
    ] = np.cos(

        2
        *
        np.pi
        *
        hour
        /
        24

    )


    x[
        "weekday"
    ] = (

        x.index.dayofweek
        /
        4

    )


    return x


# ============================================================
# 5. PREPARE DATA
#
# timestamp = bar OPEN
#
# signal:
# t bar close
#
# entry:
# t+1 open
#
# exit:
# t+2 close
#
# ============================================================

def prepare_data(
    bars
):

    frame = (

        make_features(
            bars
        )

        .replace(

            [
                np.inf,
                -np.inf,
            ],

            np.nan

        )

    )


    times = pd.Series(

        bars.index,

        index=
            bars.index,

    )


    frame[
        "entry_time"
    ] = (

        times.shift(-1)

    )


    frame[
        "label_end"
    ] = (

        times.shift(-2)

        +

        pd.Timedelta(
            minutes=15
        )

    )


    frame[
        "entry_price"
    ] = (

        bars[
            "open"
        ]

        .shift(-1)

    )


    frame[
        "exit_price"
    ] = (

        bars[
            "close"
        ]

        .shift(-2)

    )


    frame[
        "future_return"
    ] = (

        frame[
            "exit_price"
        ]

        /

        frame[
            "entry_price"
        ]

        -

        1

    )


    frame[
        "target"
    ] = (

        frame[
            "future_return"
        ]

        >
        0

    ).astype(
        int
    )


    # --------------------------------------------------------
    # 本当に連続した15m足だけ採用
    # --------------------------------------------------------

    continuous = (

        (
            times.shift(-1)
            -
            times
        )

        ==

        pd.Timedelta(
            minutes=15
        )

    ) & (

        (
            times.shift(-2)
            -
            times
        )

        ==

        pd.Timedelta(
            minutes=30
        )

    )


    frame = (

        frame.loc[
            continuous
        ]

        .dropna(

            subset=
                FEATURES
                +
                [
                    "entry_time",
                    "label_end",
                    "entry_price",
                    "exit_price",
                    "future_return",
                ]

        )

        .copy()

    )


    return frame


data = (
    prepare_data(
        df
    )
)


print()

print(
    "Usable rows:",
    len(data)
)


# ============================================================
# 6. MODEL
# ============================================================

def build_model():

    return (

        RandomForestClassifier(

            n_estimators=
                N_ESTIMATORS,

            max_depth=
                8,

            min_samples_leaf=
                30,

            max_features=
                "sqrt",

            class_weight=
                "balanced",

            random_state=
                RANDOM_STATE,

            n_jobs=
                -1,

        )

    )


# ============================================================
# 7. PREDICT
# ============================================================

def predict_frame(
    model,
    frame
):

    probability = (

        model.predict_proba(

            frame[
                FEATURES
            ]

        )

    )


    classes = list(
        model.classes_
    )


    if (
        0 not in classes
        or
        1 not in classes
    ):

        raise ValueError(

            "Training set must contain both classes."

        )


    p_up = (

        probability[
            :,
            classes.index(
                1
            )
        ]

    )


    result = (

        frame[
            [
                "entry_time",
                "label_end",
                "entry_price",
                "exit_price",
                "future_return",
            ]
        ]

        .copy()

    )


    result[
        "p_up"
    ] = (
        p_up
    )


    result[
        "confidence"
    ] = np.maximum(

        p_up,

        1
        -
        p_up

    )


    result[
        "direction"
    ] = np.where(

        p_up
        >=
        0.5,

        "BUY",

        "SELL",

    )


    result[
        "direction_correct"
    ] = (

        (
            p_up
            >=
            0.5
        )

        ==

        (
            frame[
                "future_return"
            ]
            >
            0
        )

    )


    result[
        "gross_return"
    ] = (

        frame[
            "future_return"
        ]

        *

        np.where(

            p_up
            >=
            0.5,

            1,

            -1,

        )

    )


    return result


# ============================================================
# 8. SESSION POLICY
# ============================================================

def apply_session_policy(
    frame,
    policy
):

    if frame.empty:

        return (
            frame.copy()
        )


    hour = (

        frame[
            "entry_time"
        ]

        .dt.hour

    )


    if (
        policy
        ==
        "ALL"
    ):

        mask = np.ones(

            len(frame),

            dtype=bool,

        )


    elif (
        policy
        ==
        "EXCLUDE_08_13"
    ):

        mask = ~(

            (
                hour
                >=
                8
            )

            &

            (
                hour
                <
                13
            )

        )


    elif (
        policy
        ==
        "UTC_13_24"
    ):

        mask = (

            hour
            >=
            13

        )


    elif (
        policy
        ==
        "UTC_21_24"
    ):

        mask = (

            hour
            >=
            21

        )


    else:

        raise ValueError(

            f"Unknown policy: {policy}"

        )


    return (

        frame.loc[
            mask
        ]

        .copy()

    )


# ============================================================
# 9. TRADE SELECTION
#
# IMPORTANT:
#
# entry_time < previous_exit
#
# の時だけ重複。
#
# entry_time == previous_exit
#
# は許可。
#
# ============================================================

def select_trades(
    predictions,
    threshold,
    session_policy,
    cost,
):

    candidates = (

        predictions.loc[

            predictions[
                "confidence"
            ]

            >=

            threshold

        ]

        .sort_index()

        .copy()

    )


    candidates = (

        apply_session_policy(

            candidates,

            session_policy,

        )

    )


    selected = []


    last_exit_time = None


    for row in (

        candidates.itertuples()

    ):


        # ----------------------------------------------------
        # 修正点
        #
        # equalityは許可
        # ----------------------------------------------------

        if (

            last_exit_time
            is not None

            and

            row.entry_time
            <
            last_exit_time

        ):

            continue


        selected.append(

            row.Index

        )


        last_exit_time = (

            row.label_end

        )


    trades = (

        candidates.loc[
            selected
        ]

        .copy()

    )


    trades[
        "net_return"
    ] = (

        trades[
            "gross_return"
        ]

        -

        cost

    )


    trades[
        "threshold"
    ] = (
        threshold
    )


    trades[
        "session_policy"
    ] = (
        session_policy
    )


    trades.index.name = (
        "signal_time"
    )


    return trades


# ============================================================
# 10. PERFORMANCE
# ============================================================

def profit_factor(
    returns
):

    r = np.asarray(

        returns,

        dtype=float,

    )


    gains = (

        r[
            r > 0
        ]

        .sum()

    )


    losses = (

        -r[
            r < 0
        ]

        .sum()

    )


    if losses > 0:

        return (

            gains
            /
            losses

        )


    if gains > 0:

        return (
            np.inf
        )


    return (
        np.nan
    )


def strategy_stats(
    returns
):

    r = np.asarray(

        returns,

        dtype=float,

    )


    if len(r) == 0:

        return {

            "trades":
                0,

            "win_rate":
                np.nan,

            "avg_return":
                np.nan,

            "median_return":
                np.nan,

            "profit_factor":
                np.nan,

            "max_dd":
                np.nan,

            "growth":
                np.nan,

        }


    equity = np.r_[

        1.0,

        np.cumprod(

            1
            +
            r

        )

    ]


    peak = (

        np.maximum.accumulate(

            equity

        )

    )


    dd = (

        equity
        /
        peak
        -
        1

    )


    return {

        "trades":
            len(r),

        "win_rate":
            (
                r > 0
            ).mean(),

        "avg_return":
            r.mean(),

        "median_return":
            np.median(
                r
            ),

        "profit_factor":
            profit_factor(
                r
            ),

        "max_dd":
            dd.min(),

        "growth":
            equity[-1]
            -
            1,

    }


# ============================================================
# 11. SELECT THRESHOLD
#
# Session = ALL固定
# ============================================================

def choose_threshold(
    validation_predictions
):

    rows = []


    best = None

    best_score = (
        -np.inf
    )


    for threshold in (
        THRESHOLDS
    ):


        trades = (

            select_trades(

                validation_predictions,

                threshold=
                    threshold,

                session_policy=
                    "ALL",

                cost=
                    BASE_COST,

            )

        )


        stats = (

            strategy_stats(

                trades[
                    "net_return"
                ]

            )

        )


        eligible = (

            stats[
                "trades"
            ]

            >=

            MIN_VALIDATION_TRADES

        )


        if eligible:

            score = (

                stats[
                    "avg_return"
                ]

                *

                np.sqrt(

                    stats[
                        "trades"
                    ]

                )

            )


        else:

            score = (
                np.nan
            )


        rows.append(

            {

                "threshold":
                    threshold,

                "eligible":
                    eligible,

                "score":
                    score,

                **stats,

            }

        )


        if (

            eligible

            and

            score
            >
            best_score

        ):


            best_score = (
                score
            )


            best = {

                "threshold":
                    threshold,

                "score":
                    score,

                **stats,

            }


    return (

        best,

        pd.DataFrame(
            rows
        ),

    )


# ============================================================
# 12. SELECT SESSION
#
# Thresholdは固定。
#
# ここではSessionしか変えない。
# ============================================================

def choose_session(
    validation_predictions,
    fixed_threshold,
):

    rows = []


    best = None

    best_score = (
        -np.inf
    )


    for policy in (
        SESSION_POLICIES
    ):


        trades = (

            select_trades(

                validation_predictions,

                threshold=
                    fixed_threshold,

                session_policy=
                    policy,

                cost=
                    BASE_COST,

            )

        )


        stats = (

            strategy_stats(

                trades[
                    "net_return"
                ]

            )

        )


        eligible = (

            stats[
                "trades"
            ]

            >=

            MIN_VALIDATION_TRADES

        )


        if eligible:

            score = (

                stats[
                    "avg_return"
                ]

                *

                np.sqrt(

                    stats[
                        "trades"
                    ]

                )

            )


        else:

            score = (
                np.nan
            )


        rows.append(

            {

                "threshold":
                    fixed_threshold,

                "session_policy":
                    policy,

                "eligible":
                    eligible,

                "score":
                    score,

                **stats,

            }

        )


        if (

            eligible

            and

            score
            >
            best_score

        ):


            best_score = (
                score
            )


            best = {

                "threshold":
                    fixed_threshold,

                "session_policy":
                    policy,

                "score":
                    score,

                **stats,

            }


    return (

        best,

        pd.DataFrame(
            rows
        ),

    )


# ============================================================
# 13. NESTED WALK-FORWARD
# ============================================================

years = sorted(

    data.index.year.unique()

)


annual_rows = []


baseline_trade_frames = []

session_trade_frames = []


threshold_search_frames = []

session_search_frames = []


for test_year in years:


    validation_year = (

        test_year
        -
        1

    )


    previous_years = [

        y

        for y in years

        if y
        <
        validation_year

    ]


    if (

        len(
            previous_years
        )

        <
        MIN_TRAIN_YEARS

    ):

        continue


    if validation_year not in years:

        continue


    validation_start = pd.Timestamp(

        year=
            validation_year,

        month=1,

        day=1,

        tz="UTC",

    )


    test_start = pd.Timestamp(

        year=
            test_year,

        month=1,

        day=1,

        tz="UTC",

    )


    test_end = pd.Timestamp(

        year=
            test_year
            +
            1,

        month=1,

        day=1,

        tz="UTC",

    )


    # ========================================================
    # TRAIN
    # ========================================================

    train = (

        data.loc[

            (
                data.index
                <
                validation_start
            )

            &

            (
                data[
                    "label_end"
                ]
                <=
                validation_start
            )

        ]

        .copy()

    )


    # ========================================================
    # VALIDATION
    # ========================================================

    validation = (

        data.loc[

            (
                data.index
                >=
                validation_start
            )

            &

            (
                data.index
                <
                test_start
            )

            &

            (
                data[
                    "label_end"
                ]
                <=
                test_start
            )

        ]

        .copy()

    )


    # ========================================================
    # FINAL TRAIN
    # ========================================================

    final_train = (

        data.loc[

            (
                data.index
                <
                test_start
            )

            &

            (
                data[
                    "label_end"
                ]
                <=
                test_start
            )

        ]

        .copy()

    )


    # ========================================================
    # TEST
    # ========================================================

    test = (

        data.loc[

            (
                data.index
                >=
                test_start
            )

            &

            (
                data.index
                <
                test_end
            )

            &

            (
                data[
                    "label_end"
                ]
                <=
                test_end
            )

        ]

        .copy()

    )


    if (

        len(train)
        <
        MIN_TRAIN_ROWS

        or

        len(validation)
        <
        MIN_EVAL_ROWS

        or

        len(final_train)
        <
        MIN_TRAIN_ROWS

        or

        len(test)
        <
        MIN_EVAL_ROWS

    ):

        continue


    print()

    print(
        "===================================="
    )

    print(
        f"TEST YEAR {test_year}"
    )

    print(
        "===================================="
    )


    # ========================================================
    # Train -> Validation
    # ========================================================

    model = (
        build_model()
    )


    model.fit(

        train[
            FEATURES
        ],

        train[
            "target"
        ],

    )


    validation_predictions = (

        predict_frame(

            model,

            validation,

        )

    )


    # ========================================================
    # STEP 1:
    # Threshold決定
    #
    # Session = ALL固定
    # ========================================================

    (
        threshold_choice,
        threshold_search,
    ) = choose_threshold(

        validation_predictions

    )


    if threshold_choice is None:

        print(
            "No valid threshold."
        )

        continue


    fixed_threshold = (

        threshold_choice[
            "threshold"
        ]

    )


    threshold_search[
        "test_year"
    ] = (
        test_year
    )


    threshold_search_frames.append(

        threshold_search

    )


    # ========================================================
    # STEP 2:
    # Threshold固定
    #
    # Sessionだけ選択
    # ========================================================

    (
        session_choice,
        session_search,
    ) = choose_session(

        validation_predictions,

        fixed_threshold=
            fixed_threshold,

    )


    if session_choice is None:

        print(
            "No valid session."
        )

        continue


    selected_session = (

        session_choice[
            "session_policy"
        ]

    )


    session_search[
        "test_year"
    ] = (
        test_year
    )


    session_search_frames.append(

        session_search

    )


    print(

        "Threshold:",

        fixed_threshold

    )


    print(

        "Session:",

        selected_session

    )


    # ========================================================
    # Refit using all pre-Test data
    # ========================================================

    final_model = (
        build_model()
    )


    final_model.fit(

        final_train[
            FEATURES
        ],

        final_train[
            "target"
        ],

    )


    test_predictions = (

        predict_frame(

            final_model,

            test,

        )

    )


    test_auc = (

        roc_auc_score(

            test[
                "target"
            ],

            test_predictions[
                "p_up"
            ],

        )

    )


    # ========================================================
    # BASELINE
    #
    # Threshold SAME
    # Session ALL
    # ========================================================

    base_trades = (

        select_trades(

            test_predictions,

            threshold=
                fixed_threshold,

            session_policy=
                "ALL",

            cost=
                BASE_COST,

        )

    )


    base_trades[
        "test_year"
    ] = (
        test_year
    )


    base_trades[
        "strategy"
    ] = (
        "BASELINE"
    )


    baseline_trade_frames.append(

        base_trades

    )


    base_stats = (

        strategy_stats(

            base_trades[
                "net_return"
            ]

        )

    )


    # ========================================================
    # SESSION
    #
    # Threshold SAME
    # Sessionだけ違う
    # ========================================================

    session_trades = (

        select_trades(

            test_predictions,

            threshold=
                fixed_threshold,

            session_policy=
                selected_session,

            cost=
                BASE_COST,

        )

    )


    session_trades[
        "test_year"
    ] = (
        test_year
    )


    session_trades[
        "strategy"
    ] = (
        "SESSION"
    )


    session_trade_frames.append(

        session_trades

    )


    session_stats = (

        strategy_stats(

            session_trades[
                "net_return"
            ]

        )

    )


    # ========================================================
    # Annual row
    # ========================================================

    annual_rows.append(

        {

            "test_year":
                test_year,

            "test_auc":
                test_auc,

            "threshold":
                fixed_threshold,

            "session_policy":
                selected_session,


            # BASE
            "base_trades":
                base_stats[
                    "trades"
                ],

            "base_win_rate":
                base_stats[
                    "win_rate"
                ],

            "base_avg_return":
                base_stats[
                    "avg_return"
                ],

            "base_pf":
                base_stats[
                    "profit_factor"
                ],

            "base_max_dd":
                base_stats[
                    "max_dd"
                ],


            # SESSION
            "session_trades":
                session_stats[
                    "trades"
                ],

            "session_win_rate":
                session_stats[
                    "win_rate"
                ],

            "session_avg_return":
                session_stats[
                    "avg_return"
                ],

            "session_pf":
                session_stats[
                    "profit_factor"
                ],

            "session_max_dd":
                session_stats[
                    "max_dd"
                ],

        }

    )


    print(

        "Test AUC:",

        round(
            test_auc,
            4
        )

    )


    print(

        "BASE:",

        base_stats[
            "trades"
        ],

        "trades | PF",

        round(
            base_stats[
                "profit_factor"
            ],
            3
        ),

        "| Avg",

        round(
            base_stats[
                "avg_return"
            ]
            *
            100,
            5
        ),

        "%"

    )


    print(

        "SESSION:",

        session_stats[
            "trades"
        ],

        "trades | PF",

        round(
            session_stats[
                "profit_factor"
            ],
            3
        ),

        "| Avg",

        round(
            session_stats[
                "avg_return"
            ]
            *
            100,
            5
        ),

        "%"

    )


# ============================================================
# 14. ANNUAL RESULTS
# ============================================================

annual_results = (

    pd.DataFrame(
        annual_rows
    )

)


print()

print(
    "===================================="
)

print(
    "ANNUAL PURE SESSION TEST"
)

print(
    "===================================="
)


annual_show = (

    annual_results.copy()

)


for col in [

    "threshold",

    "base_win_rate",
    "session_win_rate",

    "base_avg_return",
    "session_avg_return",

    "base_max_dd",
    "session_max_dd",

]:

    annual_show[
        col
    ] *= (
        100
    )


print(

    annual_show.to_string(
        index=False
    )

)


# ============================================================
# 15. OVERALL
# ============================================================

all_base = (

    pd.concat(
        baseline_trade_frames
    )

    .sort_index()

)


all_session = (

    pd.concat(
        session_trade_frames
    )

    .sort_index()

)


base_overall = (

    strategy_stats(

        all_base[
            "net_return"
        ]

    )

)


session_overall = (

    strategy_stats(

        all_session[
            "net_return"
        ]

    )

)


print()

print(
    "===================================="
)

print(
    "OVERALL OOS"
)

print(
    "===================================="
)


print(
    "BASELINE"
)

print(
    base_overall
)


print()

print(
    "SESSION"
)

print(
    session_overall
)


# ============================================================
# 16. SESSION POLICY FREQUENCY
# ============================================================

print()

print(
    "===================================="
)

print(
    "SESSION SELECTION FREQUENCY"
)

print(
    "===================================="
)


print(

    annual_results[
        "session_policy"
    ]

    .value_counts()

)


# ============================================================
# 17. PURE SESSION IMPROVEMENT
# ============================================================

annual_results[
    "pf_improved"
] = (

    annual_results[
        "session_pf"
    ]

    >

    annual_results[
        "base_pf"
    ]

)


annual_results[
    "avg_improved"
] = (

    annual_results[
        "session_avg_return"
    ]

    >

    annual_results[
        "base_avg_return"
    ]

)


annual_results[
    "dd_improved"
] = (

    annual_results[
        "session_max_dd"
    ]

    >

    annual_results[
        "base_max_dd"
    ]

)


print()

print(
    "===================================="
)

print(
    "PURE SESSION VALUE"
)

print(
    "===================================="
)


print(

    "PF improved:",

    annual_results[
        "pf_improved"
    ].sum(),

    "/",

    len(
        annual_results
    )

)


print(

    "Avg Return improved:",

    annual_results[
        "avg_improved"
    ].sum(),

    "/",

    len(
        annual_results
    )

)


print(

    "Max DD improved:",

    annual_results[
        "dd_improved"
    ].sum(),

    "/",

    len(
        annual_results
    )

)


print(

    "Session positive years:",

    (
        annual_results[
            "session_avg_return"
        ]
        >
        0
    ).sum(),

    "/",

    len(
        annual_results
    )

)


print(

    "Session PF > 1 years:",

    (
        annual_results[
            "session_pf"
        ]
        >
        1
    ).sum(),

    "/",

    len(
        annual_results
    )

)


# ============================================================
# 18. COST STRESS
#
# 戦略を再最適化しない。
# 同じOOS tradesにCostだけ変更。
# ============================================================

cost_rows = []


for strategy_name, frame in [

    (
        "BASELINE",
        all_base,
    ),

    (
        "SESSION",
        all_session,
    ),

]:


    for cost in (
        COST_LEVELS
    ):


        returns = (

            frame[
                "gross_return"
            ]

            -

            cost

        )


        stats = (

            strategy_stats(
                returns
            )

        )


        cost_rows.append(

            {

                "strategy":
                    strategy_name,

                "cost_pct":
                    cost
                    *
                    100,

                **stats,

            }

        )


cost_results = (

    pd.DataFrame(
        cost_rows
    )

)


cost_show = (

    cost_results.copy()

)


for col in [

    "win_rate",
    "avg_return",
    "median_return",
    "max_dd",
    "growth",

]:

    cost_show[
        col
    ] *= (
        100
    )


print()

print(
    "===================================="
)

print(
    "COST STRESS"
)

print(
    "===================================="
)


print(

    cost_show.to_string(
        index=False
    )

)


# ============================================================
# 19. YEARLY COST STABILITY
# ============================================================

year_cost_rows = []


for strategy_name, frame in [

    (
        "BASELINE",
        all_base,
    ),

    (
        "SESSION",
        all_session,
    ),

]:


    for year, year_frame in (

        frame.groupby(
            "test_year"
        )

    ):


        for cost in (
            COST_LEVELS
        ):


            returns = (

                year_frame[
                    "gross_return"
                ]

                -

                cost

            )


            stats = (

                strategy_stats(
                    returns
                )

            )


            year_cost_rows.append(

                {

                    "strategy":
                        strategy_name,

                    "test_year":
                        year,

                    "cost_pct":
                        cost
                        *
                        100,

                    **stats,

                }

            )


year_cost_results = (

    pd.DataFrame(
        year_cost_rows
    )

)


stability_rows = []


for (

    strategy_name,
    cost_pct

), group in (

    year_cost_results.groupby(

        [
            "strategy",
            "cost_pct",
        ]

    )

):


    stability_rows.append(

        {

            "strategy":
                strategy_name,

            "cost_pct":
                cost_pct,

            "years":
                len(
                    group
                ),

            "positive_years":
                (
                    group[
                        "avg_return"
                    ]
                    >
                    0
                ).sum(),

            "pf_above_1_years":
                (
                    group[
                        "profit_factor"
                    ]
                    >
                    1
                ).sum(),

            "median_year_pf":
                group[
                    "profit_factor"
                ].median(),

            "minimum_year_pf":
                group[
                    "profit_factor"
                ].min(),

        }

    )


cost_stability = (

    pd.DataFrame(
        stability_rows
    )

)


print()

print(
    "===================================="
)

print(
    "COST YEAR STABILITY"
)

print(
    "===================================="
)


print(

    cost_stability.to_string(
        index=False
    )

)


# ============================================================
# 20. BREAK-EVEN COST
# ============================================================

base_gross_mean = (

    all_base[
        "gross_return"
    ].mean()

)


session_gross_mean = (

    all_session[
        "gross_return"
    ].mean()

)


print()

print(
    "===================================="
)

print(
    "APPROX BREAK-EVEN COST"
)

print(
    "===================================="
)


print(

    "BASELINE:",

    base_gross_mean
    *
    100,

    "%"

)


print(

    "SESSION:",

    session_gross_mean
    *
    100,

    "%"

)


# ============================================================
# 21. AUTOMATIC DECISION
# ============================================================

print()

print(
    "===================================="
)

print(
    "AUTOMATIC DECISION"
)

print(
    "===================================="
)


n_years = (
    len(
        annual_results
    )
)


pf_better = (

    annual_results[
        "pf_improved"
    ].sum()

)


avg_better = (

    annual_results[
        "avg_improved"
    ].sum()

)


session_pf = (

    session_overall[
        "profit_factor"
    ]

)


base_pf = (

    base_overall[
        "profit_factor"
    ]

)


session_avg = (

    session_overall[
        "avg_return"
    ]

)


base_avg = (

    base_overall[
        "avg_return"
    ]

)


print(

    "Overall Base PF:",

    base_pf

)


print(

    "Overall Session PF:",

    session_pf

)


print(

    "Overall Base Avg Return:",

    base_avg
    *
    100,

    "%"

)


print(

    "Overall Session Avg Return:",

    session_avg
    *
    100,

    "%"

)


print()

print(

    "PF better years:",

    pf_better,

    "/",

    n_years

)


print(

    "Avg better years:",

    avg_better,

    "/",

    n_years

)


print()


# Session採用条件をかなり厳しめにする
if (

    session_pf
    >
    base_pf

    and

    session_avg
    >
    base_avg

    and

    pf_better
    >=
    4

    and

    avg_better
    >=
    4

):


    print(

        "判定: SESSION FILTER HAS CLEAR OOS VALUE"

    )


    print(

        "Session Filterを正式戦略候補として残せます。"

    )


else:


    print(

        "判定: SESSION FILTER DOES NOT ADD CLEAR ENOUGH VALUE"

    )


    print(

        "時間帯差は研究結果として残しますが、"

    )


    print(

        "Hard Session Filterを無理に本戦略へ追加する必要はありません。"

    )


    print(

        "次はTP / SL / Max Hold検証へ進むのが合理的です。"

    )


# ============================================================
# 22. GRAPH - PF
# ============================================================

plt.figure(

    figsize=(
        9,
        5
    )

)


plt.plot(

    annual_results[
        "test_year"
    ],

    annual_results[
        "base_pf"
    ],

    marker="o",

    label=
        "Baseline",

)


plt.plot(

    annual_results[
        "test_year"
    ],

    annual_results[
        "session_pf"
    ],

    marker="o",

    label=
        "Session",

)


plt.axhline(

    1,

    linewidth=1,

)


plt.xlabel(
    "Test Year"
)

plt.ylabel(
    "Profit Factor"
)

plt.title(
    "Pure Session Incremental Value"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# 23. GRAPH - Avg Return
# ============================================================

plt.figure(

    figsize=(
        9,
        5
    )

)


plt.plot(

    annual_results[
        "test_year"
    ],

    annual_results[
        "base_avg_return"
    ]
    *
    100,

    marker="o",

    label=
        "Baseline",

)


plt.plot(

    annual_results[
        "test_year"
    ],

    annual_results[
        "session_avg_return"
    ]
    *
    100,

    marker="o",

    label=
        "Session",

)


plt.axhline(

    0,

    linewidth=1,

)


plt.xlabel(
    "Test Year"
)

plt.ylabel(
    "Average Return (%)"
)

plt.title(
    "OOS Average Return"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# 24. SAVE
# ============================================================

annual_results.to_csv(

    OUTPUT_DIR
    /
    "annual_pure_session_test.csv",

    index=False,

)


all_base.to_csv(

    OUTPUT_DIR
    /
    "baseline_oos_trades.csv",

)


all_session.to_csv(

    OUTPUT_DIR
    /
    "session_oos_trades.csv",

)


cost_results.to_csv(

    OUTPUT_DIR
    /
    "cost_stress.csv",

    index=False,

)


year_cost_results.to_csv(

    OUTPUT_DIR
    /
    "cost_stress_yearly.csv",

    index=False,

)


cost_stability.to_csv(

    OUTPUT_DIR
    /
    "cost_stability.csv",

    index=False,

)


if threshold_search_frames:

    pd.concat(

        threshold_search_frames,

        ignore_index=True,

    ).to_csv(

        OUTPUT_DIR
        /
        "threshold_validation_search.csv",

        index=False,

    )


if session_search_frames:

    pd.concat(

        session_search_frames,

        ignore_index=True,

    ).to_csv(

        OUTPUT_DIR
        /
        "session_validation_search.csv",

        index=False,

    )


print()

print(
    "===================================="
)

print(
    "FINISHED"
)

print(
    "===================================="
)


print(

    OUTPUT_DIR.resolve()

)


print()

print(
    "結果で見せてほしい場所:"
)

print(
    "1. ANNUAL PURE SESSION TEST"
)

print(
    "2. OVERALL OOS"
)

print(
    "3. SESSION SELECTION FREQUENCY"
)

print(
    "4. PURE SESSION VALUE"
)

print(
    "5. COST STRESS"
)

print(
    "6. COST YEAR STABILITY"
)

print(
    "7. AUTOMATIC DECISION"
)
